In [61]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

import pandas as pd
import optuna

In [ ]:
df = pd.read_csv('../data/processed/train.csv')

X_train = df.drop(columns='Survived')
y_train = df['Survived']

In [63]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# LightGBM

In [64]:
def objective_lgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 4, 64, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    }

    model = LGBMClassifier(
        **params,
        subsample_freq=1,
        random_state=42,
        n_jobs=1,
        verbose=-1,
    )

    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return scores.mean()

In [65]:
study_lgbm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'Accuracy модели: {study_lgbm.best_value}')
print(f'Лучшие параметры: {study_lgbm.best_params}')

[I 2026-10-05 19:56:37,449] A new study created in memory with name: no-name-a574ba61-0941-42e2-9b4a-ae5832c69820
Best trial: 0. Best value: 0.80806:   2%|▏         | 1/50 [00:04<03:36,  4.42s/it]

[I 2026-10-05 19:56:41,880] Trial 0 finished with value: 0.8080597577051032 and parameters: {'num_leaves': 10, 'learning_rate': 0.17254716573280354, 'n_estimators': 500, 'min_child_samples': 32, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_lambda': 0.0017073967431528124}. Best is trial 0 with value: 0.8080597577051032.


Best trial: 1. Best value: 0.811418:   4%|▍         | 2/50 [00:06<02:40,  3.34s/it]

[I 2026-10-05 19:56:44,454] Trial 1 finished with value: 0.8114179900822295 and parameters: {'num_leaves': 44, 'learning_rate': 0.06054365855469246, 'n_estimators': 450, 'min_child_samples': 5, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'reg_lambda': 0.0070689749506246055}. Best is trial 1 with value: 0.8114179900822295.


Best trial: 3. Best value: 0.834982:   6%|▌         | 3/50 [00:09<02:20,  3.00s/it]

...

[I 2026-10-05 19:56:54,281] Trial 45 finished with value: 0.84509446990145 and parameters: {'num_leaves': 31, 'learning_rate': 0.016649521420475325, 'n_estimators': 600, 'min_child_samples': 20, 'subsample': 0.6141612556624214, 'colsample_bytree': 0.914016618434118, 'reg_lambda': 0.8927891197217337}. Best is trial 10 with value: 0.8507124474295399.
[I 2026-10-05 19:56:54,404] Trial 46 finished with value: 0.8305065595380077 and parameters: {'num_leaves': 8, 'learning_rate': 0.01169346637640889, 'n_estimators': 400, 'min_child_samples': 13, 'subsample': 0.6824202025124283, 'colsample_bytree': 0.8814745951702188, 'reg_lambda': 5.67606941215454}. Best is trial 10 with value: 0.8507124474295399.


Best trial: 10. Best value: 0.850712:  98%|█████████▊| 49/50 [00:17<00:00,  5.69it/s]

[I 2026-10-05 19:56:54,645] Trial 47 finished with value: 0.8451007469713139 and parameters: {'num_leaves': 24, 'learning_rate': 0.010008560788950408, 'n_estimators': 500, 'min_child_samples': 16, 'subsample': 0.90026802737786, 'colsample_bytree': 0.8997670967201322, 'reg_lambda': 0.09459431602088948}. Best is trial 10 with value: 0.8507124474295399.
[I 2026-10-05 19:56:54,810] Trial 48 finished with value: 0.8507124474295399 and parameters: {'num_leaves': 12, 'learning_rate': 0.019569953039137614, 'n_estimators': 550, 'min_child_samples': 7, 'subsample': 0.6515767098485051, 'colsample_bytree': 0.9349915869568574, 'reg_lambda': 3.215861317027293}. Best is trial 10 with value: 0.8507124474295399.


Best trial: 10. Best value: 0.850712: 100%|██████████| 50/50 [00:17<00:00,  2.87it/s]

[I 2026-10-05 19:56:54,881] Trial 49 finished with value: 0.8451007469713137 and parameters: {'num_leaves': 11, 'learning_rate': 0.02734774295492769, 'n_estimators': 100, 'min_child_samples': 5, 'subsample': 0.6374898265855397, 'colsample_bytree': 0.9481002630895989, 'reg_lambda': 0.4596318966168473}. Best is trial 10 with value: 0.8507124474295399.
Accuracy модели: 0.8507124474295399
Лучшие параметры: {'num_leaves': 23, 'learning_rate': 0.010181283131439643, 'n_estimators': 600, 'min_child_samples': 16, 'subsample': 0.6058163500649457, 'colsample_bytree': 0.9630659181130071, 'reg_lambda': 0.834579246362133}


# CatBoost

In [66]:
def objective_cat(trial):
    params = {
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'iterations': trial.suggest_int('iterations', 200, 800, step=100),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
    }

    model = CatBoostClassifier(
        **params,
        random_seed=42,
        thread_count=1,
        verbose=0,
        allow_writing_files=False,
    )

    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return scores.mean()

In [67]:
study_cat = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study_cat.optimize(objective_cat, n_trials=50, show_progress_bar=True)

print(f'Accuracy модели: {study_cat.best_value}')
print(f'Лучшие параметры: {study_cat.best_params}')

[I 2026-10-05 19:56:54,921] A new study created in memory with name: no-name-ed89bf57-1b92-4580-9fdc-63d19afb02ee
Best trial: 0. Best value: 0.830513:   2%|▏         | 1/50 [00:01<00:55,  1.13s/it]

[I 2026-10-05 19:56:56,054] Trial 0 finished with value: 0.8305128366078713 and parameters: {'depth': 5, 'learning_rate': 0.17254716573280354, 'iterations': 700, 'l2_leaf_reg': 3.968793330444372}. Best is trial 0 with value: 0.8305128366078713.


Best trial: 0. Best value: 0.830513:   4%|▍         | 2/50 [00:01<00:35,  1.36it/s]

[I 2026-10-05 19:56:56,513] Trial 1 finished with value: 0.828278199736363 and parameters: {'depth': 3, 'learning_rate': 0.015957084694148364, 'iterations': 200, 'l2_leaf_reg': 7.348118405270449}. Best is trial 0 with value: 0.8305128366078713.


Best trial: 2. Best value: 0.839483:   6%|▌         | 3/50 [00:02<00:29,  1.59it/s]

...

[I 2026-10-05 19:57:38,190] Trial 47 finished with value: 0.8349883874207519 and parameters: {'depth': 3, 'learning_rate': 0.05313090185879512, 'iterations': 500, 'l2_leaf_reg': 3.7919945187763062}. Best is trial 21 with value: 0.8495951289937856.


Best trial: 21. Best value: 0.849595:  98%|█████████▊| 49/50 [00:43<00:00,  1.24it/s]

[I 2026-10-05 19:57:38,683] Trial 48 finished with value: 0.8451070240411775 and parameters: {'depth': 6, 'learning_rate': 0.0699078429390049, 'iterations': 300, 'l2_leaf_reg': 9.254625079484272}. Best is trial 21 with value: 0.8495951289937856.


Best trial: 21. Best value: 0.849595: 100%|██████████| 50/50 [00:45<00:00,  1.11it/s]

[I 2026-10-05 19:57:40,041] Trial 49 finished with value: 0.8473479379825497 and parameters: {'depth': 8, 'learning_rate': 0.04771485595266047, 'iterations': 400, 'l2_leaf_reg': 7.199867045103297}. Best is trial 21 with value: 0.8495951289937856.
Accuracy модели: 0.8495951289937856
Лучшие параметры: {'depth': 7, 'learning_rate': 0.063657807779867, 'iterations': 400, 'l2_leaf_reg': 6.14844310356552}


# RandomForest

In [68]:
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_float('max_features', 0.3, 1.0),
    }

    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=1,
    )

    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return scores.mean()

In [69]:
study_rf = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=True)

print(f'Accuracy модели: {study_rf.best_value}')
print(f'Лучшие параметры: {study_rf.best_params}')

[I 2026-10-05 19:57:40,086] A new study created in memory with name: no-name-f53f887b-640b-42fa-8b44-41f1d10d9de1
Best trial: 0. Best value: 0.83163:   2%|▏         | 1/50 [00:01<00:58,  1.19s/it]

[I 2026-10-05 19:57:41,278] Trial 0 finished with value: 0.8316301550436256 and parameters: {'n_estimators': 400, 'max_depth': 12, 'min_samples_leaf': 8, 'max_features': 0.7190609389379257}. Best is trial 0 with value: 0.8316301550436256.


Best trial: 0. Best value: 0.83163:   4%|▍         | 2/50 [00:01<00:46,  1.04it/s]

[I 2026-10-05 19:57:42,086] Trial 1 finished with value: 0.8293955181721173 and parameters: {'n_estimators': 300, 'max_depth': 4, 'min_samples_leaf': 1, 'max_features': 0.9063233020424546}. Best is trial 0 with value: 0.8316301550436256.


Best trial: 2. Best value: 0.841736:   6%|▌         | 3/50 [00:03<00:58,  1.25s/it]

...

[I 2026-10-05 19:58:54,655] Trial 47 finished with value: 0.8327537505492437 and parameters: {'n_estimators': 400, 'max_depth': 7, 'min_samples_leaf': 7, 'max_features': 0.7622064978750105}. Best is trial 45 with value: 0.8451070240411775.


Best trial: 45. Best value: 0.845107:  98%|█████████▊| 49/50 [01:15<00:01,  1.05s/it]

[I 2026-10-05 19:58:55,926] Trial 48 finished with value: 0.8428535559600776 and parameters: {'n_estimators': 500, 'max_depth': 6, 'min_samples_leaf': 3, 'max_features': 0.8212157047311581}. Best is trial 45 with value: 0.8451070240411775.


Best trial: 45. Best value: 0.845107: 100%|██████████| 50/50 [01:17<00:00,  1.54s/it]

[I 2026-10-05 19:58:57,164] Trial 49 finished with value: 0.8316238779737619 and parameters: {'n_estimators': 500, 'max_depth': 5, 'min_samples_leaf': 2, 'max_features': 0.875033633742304}. Best is trial 45 with value: 0.8451070240411775.
Accuracy модели: 0.8451070240411775
Лучшие параметры: {'n_estimators': 500, 'max_depth': 7, 'min_samples_leaf': 1, 'max_features': 0.8737759468154714}


# Final

In [ ]:
test_df = pd.read_csv('../data/processed/test.csv')
raw_test_df = pd.read_csv('../data/raw/test.csv')

418 418


In [ ]:
model = LGBMClassifier(**study_lgbm.best_params, random_seed=42)
model.fit(X_train, y_train)

predictions = model.predict(test_df)

submission = pd.DataFrame({
    'PassengerId': raw_test_df['PassengerId'],
    'Survived': predictions
})

submission.to_csv('./submissionlgbm.csv', index=False)

In [78]:
results = pd.DataFrame({
    'Name': ['LightGBM', 'CatBoost', 'RandomForest'],
    'CV': [study_lgbm.best_value, study_cat.best_value, study_rf.best_value],
    'LB': [0.77033, None, None]
})

results

,Name,CV,LB
0,LightGBM,0.850712,0.77033
1,CatBoost,0.849595,NaN
2,RandomForest,0.845107,NaN
